# Prediksi Kecepatan Lalu Lintas Beijing

Notebook ini membangun ulang submission final dari nol. Alurnya baca data, petakan teks event ke segmen jalan, bangun fitur, latih 4 varian LightGBM, gabungkan dengan bobot least squares, lalu tulis file submission. Dependensi di luar bawaan Anaconda cuma lightgbm dan pypinyin. Runtime total sekitar 30 sampai 40 menit di CPU.

In [1]:
import os, re, json, gc, time
import numpy as np
import pandas as pd
import lightgbm as lgb
from collections import Counter, defaultdict
from pypinyin import lazy_pinyin

DATA = "data"
OUT = "outputs"
os.makedirs(OUT, exist_ok=True)

HIST = 15
HORIZONS = (5, 10, 15)
N_SEG = 1260
VAL_STEPS = 720

## Baca data

Train terdiri dari dua blok kontinu. Blok m1 berisi 31 hari dan m2 berisi 14 hari dengan interval 4 menit per baris. Test berisi 540 window sepanjang 15 langkah alias 1 jam. Kecepatan bernilai 0 artinya sensor mati, bukan macet, dan ini penting untuk penanganan nanti.

In [2]:
speed = {
    "m1": np.load(f"{DATA}/train/train_speed_m1_1_11160.npy"),
    "m2": np.load(f"{DATA}/train/train_speed_m2_1_5039.npy"),
}
texts = {
    "m1": json.load(open(f"{DATA}/train/train_text_m1_1_11160.json", encoding="utf-8")),
    "m2": json.load(open(f"{DATA}/train/train_text_m2_1_5039.json", encoding="utf-8")),
    "test": json.load(open(f"{DATA}/test/test_texts.json", encoding="utf-8")),
}
test_hist = np.load(f"{DATA}/test/test_X_hist.npy").astype(np.float32)
adj = np.load(f"{DATA}/static/matrix.npy").astype(np.float32)
roads = json.load(open(f"{DATA}/static/Roads1260.json", encoding="utf-8"))

print(speed["m1"].shape, speed["m2"].shape, test_hist.shape, adj.shape, len(roads))

(11160, 1260) (5039, 1260) (540, 15, 1260) (1260, 1260) 1260


## Window dan validasi

Sampel train dibentuk dari sliding window 15 langkah dengan target 5, 10, dan 15 langkah setelah window. Dua hari terakhir tiap blok disisihkan jadi holdout. Window train diberi jarak 30 langkah dari zona holdout supaya tidak ada kebocoran waktu.

In [3]:
def window_splits(block, stride=1):
    T = speed[block].shape[0]
    val_start = T - VAL_STEPS
    train_ends = np.arange(HIST - 1, val_start - 29, stride)
    val_ends = np.arange(val_start, T - 15)
    return train_ends, val_ends

def mse(pred, truth):
    return float(np.mean((pred.astype(np.float64) - truth.astype(np.float64)) ** 2))

{b: [len(a) for a in window_splits(b, 2)] for b in ("m1", "m2")}

{'m1': [5199, 705], 'm2': [2138, 705]}

## Petakan nama jalan ke teks event

Teks event menyebut jalan dalam bahasa Inggris sementara metadata segmen memakai nama hanzi. Kosakatanya kecil, sekitar 190 nama unik di tiap sisi, jadi bisa dipetakan sekali di awal. Tiap nama hanzi diubah jadi beberapa kandidat token lewat pinyin dan kamus arah, angka, serta akhiran jalan, lalu dicocokkan ke lokasi Inggris pakai skor overlap token dengan bonus kesamaan urutan. Coverage akhir sekitar 99.5% dari seluruh penyebutan lokasi. Hasil mapping dipakai untuk membuat flag event per langkah per segmen untuk 6 kelas kejadian.

In [4]:
seg_name = []
for item in roads:
    subs = item if isinstance(item, list) else [item]
    names = Counter(s.get("roadName", "") for s in subs)
    seg_name.append(names.most_common(1)[0][0])
cn_names = sorted(set(seg_name))

DICT = [
    ("高速公路", ["expressway"]), ("快速路", ["expressway"]), ("机场第二", ["airport", "second"]),
    ("高速", ["expressway"]), ("国道", ["national", "highway"]), ("辅路", ["auxiliary", "road"]),
    ("入口", ["entrance"]), ("出口", ["exit"]), ("大街", ["street"]),
    ("二环", ["second", "ring"]), ("三环", ["third", "ring"]), ("四环", ["fourth", "ring"]),
    ("五环", ["fifth", "ring"]), ("机场", ["airport"]),
    ("东", ["east"]), ("西", ["west"]), ("南", ["south"]), ("北", ["north"]), ("中", ["middle"]),
    ("内", ["inner"]), ("外", ["outer"]), ("环", ["ring"]),
    ("路", ["road"]), ("桥", ["bridge"]), ("街", ["street"]),
]
CITY = {"京": "beijing", "津": "tianjin", "台": "taiwan", "藏": "tibet", "承": "chengde",
        "沪": "shanghai", "港": "hong kong", "澳": "macao", "哈": "harbin", "石": "shijiazhuang"}
GENERIC = {"road", "street", "bridge", "expressway", "ring", "east", "west", "south", "north",
           "middle", "side", "national", "highway", "inner", "outer",
           "first", "second", "third", "fourth", "fifth", "airport"}
PINYIN_PATCH = {"zhaoyang": "chaoyang"}
DIRS = set("东西南北中内外")
SUFFIXES = ["高速公路入口", "高速公路出口", "高速入口", "高速出口", "高速公路", "快速路",
            "国道", "高速", "大街", "辅路", "入口", "出口", "路", "桥", "街"]

def _clean(tok, patch):
    tok = re.sub(r"^[sg]\d+", "", tok.lower())
    if patch:
        for k, v in PINYIN_PATCH.items():
            if tok.startswith(k):
                tok = v + tok[len(k):]
    return tok

def cn_tokens(name, expand_city, dirs_mode, patch=True):
    if dirs_mode == "suffix":
        core, suf_toks = name, []
        changed = True
        while changed:
            changed = False
            for sfx in SUFFIXES:
                if core.endswith(sfx):
                    for k, v in DICT:
                        if k == sfx:
                            suf_toks = v + suf_toks
                            break
                    else:
                        sub = [t for part in (sfx[:-2], sfx[-2:]) for k, v in DICT
                               if k == part for t in v]
                        suf_toks = sub + suf_toks
                    core = core[: -len(sfx)]
                    changed = True
                    break
        toks = []
        if core:
            t = _clean("".join(lazy_pinyin(core)), patch)
            if t:
                toks.append(t)
        return toks + suf_toks
    toks, buf = [], []
    i = 0
    def flush():
        if buf:
            t = _clean("".join(lazy_pinyin("".join(buf))), patch)
            if t:
                toks.append(t)
            buf.clear()
    while i < len(name):
        ch = name[i]
        hit = None
        for k, v in DICT:
            if name.startswith(k, i):
                hit = (k, v)
                break
        if hit and len(hit[0]) == 1 and ch in DIRS:
            translate = dirs_mode == "all" or (dirs_mode == "first" and i > 0)
            if not translate:
                hit = None
        if hit:
            flush()
            toks.extend(hit[1])
            i += len(hit[0])
        elif expand_city and ch in CITY:
            flush()
            toks.extend(CITY[ch].split())
            i += 1
        else:
            buf.append(ch)
            i += 1
    flush()
    return [t for t in toks if t]

def en_tokens(loc):
    loc = re.sub(r"^(entrance|exit) of (.+)$", r"\2 \1", loc.lower())
    toks = [t for t in re.split(r"[\s\-]+", loc) if t]
    return [t for t in toks if not re.fullmatch(r"[sg]\d+", t)]

def _lcs(a, b):
    m, n = len(a), len(b)
    dp = [[0] * (n + 1) for _ in range(m + 1)]
    for i in range(m):
        for j in range(n):
            dp[i + 1][j + 1] = dp[i][j] + 1 if a[i] == b[j] else max(dp[i][j + 1], dp[i + 1][j])
    return dp[m][n]

def _w(tok):
    return 1.0 if tok in GENERIC else 3.0

def score(cn_t, en_t):
    used = [False] * len(en_t)
    matched = 0.0
    for c in cn_t:
        best, bi = 0.0, -1
        for j, e in enumerate(en_t):
            if used[j]:
                continue
            if c == e:
                v = min(_w(c), _w(e))
            elif len(c) >= 4 and len(e) >= 4 and (c.startswith(e) or e.startswith(c)):
                v = 1.5
            else:
                v = 0.0
            if v > best:
                best, bi = v, j
        if bi >= 0:
            used[bi] = True
            matched += best
    denom = max(sum(_w(t) for t in cn_t), sum(_w(t) for t in en_t), 1e-9)
    return matched / denom + 0.01 * _lcs(cn_t, en_t)

locs = Counter()
for block in ("m1", "m2", "test"):
    for txt in texts[block].values():
        for sent in txt.split("."):
            m = re.search(r"\bon (.+)$", sent.strip())
            if m:
                locs[m.group(1).strip()] += 1
en_locs = sorted(locs)
en_tok = {l: en_tokens(l) for l in en_locs}

mapping = {}
for cn in cn_names:
    cands = [cn_tokens(cn, city, dm, patch) for city in (False, True)
             for dm in ("all", "none", "first", "suffix") for patch in (True, False)]
    best = (-9, None)
    for l in en_locs:
        v = max(score(c, en_tok[l]) for c in cands)
        if v > best[0]:
            best = (v, l)
    if best[0] >= 0.55:
        mapping[cn] = best[1]

en2cn = defaultdict(set)
for cn, en in mapping.items():
    en2cn[en].add(cn)
loc2segs = {en: {i for i, n in enumerate(seg_name) if n in cns} for en, cns in en2cn.items()}
covered = sum(locs[l] for l in loc2segs)
print(f"{len(mapping)} dari {len(cn_names)} nama terpetakan, coverage {covered / sum(locs.values()) * 100:.1f}%")

184 dari 191 nama terpetakan, coverage 99.5%


In [5]:
EV_CLASSES = ["closure", "accident", "construction", "control", "prohibit", "other"]

def sent_classes(prefix):
    p = prefix.lower()
    out = set()
    if "closure" in p:
        out.add("closure")
    if "accident" in p:
        out.add("accident")
    if "construction" in p:
        out.add("construction")
    if "traffic control" in p:
        out.add("control")
    if "prohibit" in p:
        out.add("prohibit")
    if not out:
        out.add("other")
    return out

def parse_text(txt):
    res = {c: set() for c in EV_CLASSES}
    for sent in txt.split("."):
        sent = sent.strip()
        if not sent:
            continue
        m = re.search(r"\bon (.+)$", sent)
        if not m:
            continue
        loc = m.group(1).strip()
        segs = loc2segs.get(loc)
        if not segs:
            loc2 = re.sub(r"^(entrance|exit) of ", "", loc)
            loc2 = re.sub(r"\s+(entrance|exit)$", "", loc2)
            segs = loc2segs.get(loc2)
        if segs:
            for c in sent_classes(sent[: m.start()]):
                res[c] |= segs
    return res

events = {}
for block, T in (("m1", 11160), ("m2", 5039), ("test", 540)):
    mats = {c: np.zeros((T, N_SEG), dtype=np.uint8) for c in EV_CLASSES}
    for t in range(T):
        key = f"test_{t:05d}" if block == "test" else f"{block}_{t + 1}"
        for c, segs in parse_text(texts[block][key]).items():
            if segs:
                mats[c][t, list(segs)] = 1
    events[block] = mats

{b: int(events[b]["accident"].sum()) for b in events}

{'m1': 879624, 'm2': 194025, 'test': 42516}

## Fitur

Satu baris data adalah kombinasi window dan segmen. Fiturnya statistik history 15 langkah, deviasi dari rata-rata window dan rata-rata historis segmen sebagai sinyal mean reversion, agregat tetangga 1 hop dan 2 hop di graf berarah, kondisi rata-rata jaringan, metadata jalan, indikator sensor mati, hitungan kata kunci teks global, dan flag event per segmen. Totalnya 45 kolom.

In [6]:
stat = np.zeros((N_SEG, 6), dtype=np.float32)
for j, item in enumerate(roads):
    subs = item if isinstance(item, list) else [item]
    stat[j, 0] = subs[0].get("roadclass", -1)
    stat[j, 1] = subs[0].get("formway", -1)
    stat[j, 2] = sum(s.get("length", 0) for s in subs)
    stat[j, 3] = len(subs)
stat[:, 4] = adj.sum(axis=0)
stat[:, 5] = adj.sum(axis=1)

Ain = (adj / np.maximum(adj.sum(axis=0, keepdims=True), 1)).astype(np.float32)
Aout = (adj / np.maximum(adj.sum(axis=1, keepdims=True), 1)).T.astype(np.float32)
A2 = ((adj @ adj) > 0).astype(np.float32)
Ain2 = (A2 / np.maximum(A2.sum(axis=0, keepdims=True), 1)).astype(np.float32)
Aout2 = (A2 / np.maximum(A2.sum(axis=1, keepdims=True), 1)).T.astype(np.float32)

seg_mean = speed["m1"][: speed["m1"].shape[0] - VAL_STEPS - 30].mean(axis=0).astype(np.float32)

KEYWORDS = ["road closure", "construction", "accident", "announcement", "prohibit", "congest"]

def text_counts(text_list):
    out = np.zeros((len(text_list), len(KEYWORDS) + 1), dtype=np.float32)
    for i, t in enumerate(text_list):
        tl = t.lower()
        for k, kw in enumerate(KEYWORDS):
            out[i, k] = tl.count(kw)
        out[i, -1] = tl.count(".")
    return out

FEAT_NAMES = (
    ["last", "mean15", "std15", "mean5", "mean3", "diff1", "diff4",
     "last_m_mean15", "min15", "max15", "zfrac", "last_is_zero", "allzero",
     "segmean", "last_m_segmean",
     "in_nb_last", "out_nb_last", "in_nb_diff4", "out_nb_diff4",
     "in2_nb_last", "out2_nb_last",
     "net_last", "net_diff4",
     "roadclass", "formway", "length", "n_subsegs", "indeg", "outdeg", "seg_id"]
    + [f"txt_{k.split()[-1]}" for k in KEYWORDS] + ["txt_nsent"]
)
EV_NAMES = [f"ev_{c}" for c in EV_CLASSES] + ["in_nb_accident", "out_nb_accident"]
ALL_NAMES = FEAT_NAMES + EV_NAMES

def build_features(hist, txt, ev, chunk=512):
    W = hist.shape[0]
    X = np.empty((W * N_SEG, len(ALL_NAMES)), dtype=np.float32)
    for c0 in range(0, W, chunk):
        c1 = min(c0 + chunk, W)
        h = hist[c0:c1]
        C = h.shape[0]
        last = h[:, -1, :]
        mean15 = h.mean(axis=1)
        mean5 = h[:, -5:, :].mean(axis=1)
        mean3 = h[:, -3:, :].mean(axis=1)
        diff1 = last - h[:, -2, :]
        diff4 = last - h[:, -5, :]
        zfrac = (h == 0).mean(axis=1)
        nz = last > 0
        net_last = np.where(nz.any(axis=1),
                            (last * nz).sum(axis=1) / np.maximum(nz.sum(axis=1), 1), 0)
        net_d4 = (diff4 * nz).sum(axis=1) / np.maximum(nz.sum(axis=1), 1)
        acc = ev["accident"][c0:c1].astype(np.float32)
        cols = [last, mean15, h.std(axis=1), mean5, mean3, diff1, diff4,
                last - mean15, h.min(axis=1), h.max(axis=1), zfrac,
                (last == 0).astype(np.float32), (zfrac == 1).astype(np.float32),
                np.broadcast_to(seg_mean, (C, N_SEG)), last - seg_mean,
                last @ Ain, last @ Aout, diff4 @ Ain, diff4 @ Aout,
                last @ Ain2, last @ Aout2,
                np.broadcast_to(net_last[:, None], (C, N_SEG)),
                np.broadcast_to(net_d4[:, None], (C, N_SEG))]
        for k in range(6):
            cols.append(np.broadcast_to(stat[:, k], (C, N_SEG)))
        cols.append(np.broadcast_to(np.arange(N_SEG, dtype=np.float32), (C, N_SEG)))
        for k in range(txt.shape[1]):
            cols.append(np.broadcast_to(txt[c0:c1, k][:, None], (C, N_SEG)))
        for c in EV_CLASSES:
            cols.append(ev[c][c0:c1].astype(np.float32))
        cols.append(acc @ Ain)
        cols.append(acc @ Aout)
        X[c0 * N_SEG : c1 * N_SEG] = np.stack(cols, axis=-1).reshape(C * N_SEG, -1)
    return X

def build_block(block, ends):
    s = speed[block]
    hist = s[ends[:, None] + np.arange(-HIST + 1, 1)[None, :]]
    txt = text_counts([texts[block][f"{block}_{e + 1}"] for e in ends])
    ev = {c: events[block][c][ends] for c in EV_CLASSES}
    X = build_features(hist, txt, ev)
    last = s[ends].reshape(-1)
    ys = {h: s[ends + h].reshape(-1) for h in HORIZONS}
    return X, last, ys

def build_split(kind, stride=1):
    Xs, lasts, ys = [], [], {h: [] for h in HORIZONS}
    for b in ("m1", "m2"):
        tr, va = window_splits(b, stride)
        X, last, y = build_block(b, tr if kind == "train" else va)
        Xs.append(X)
        lasts.append(last)
        for h in HORIZONS:
            ys[h].append(y[h])
    return (np.concatenate(Xs), np.concatenate(lasts),
            {h: np.concatenate(v) for h, v in ys.items()})

## Latih 4 varian model

Model memprediksi residual terhadap nilai terakhir, bukan kecepatan mentah, karena baseline persistence sudah kuat. Tiap horizon dilatih terpisah. Empat varian memakai stride, subset fitur, dan kapasitas berbeda supaya hasil gabungannya beragam. Baris yang windownya nol semua berasal dari sensor mati sehingga hanya 15% yang diikutkan.

In [7]:
COLS_V1 = [i for i, n in enumerate(ALL_NAMES)
           if n in FEAT_NAMES and n not in ("in2_nb_last", "out2_nb_last")]
COLS_V2 = [i for i, n in enumerate(ALL_NAMES) if n in FEAT_NAMES]
COLS_V4 = list(range(len(ALL_NAMES)))

BASE = dict(objective="regression", metric="l2", feature_fraction=0.85, bagging_fraction=0.8,
            bagging_freq=1, max_bin=255, force_row_wise=True, num_threads=16,
            verbosity=-1, seed=42)

VARIANTS = {
    "v1": dict(stride=4, cols=COLS_V1, num_leaves=96, learning_rate=0.07, min_data=300,
               rounds=700, patience=60),
    "v2": dict(stride=2, cols=COLS_V2, num_leaves=96, learning_rate=0.07, min_data=300,
               rounds=900, patience=100),
    "v3": dict(stride=2, cols=COLS_V2, num_leaves=255, learning_rate=0.05, min_data=150,
               rounds=1400, patience=120),
    "v4": dict(stride=2, cols=COLS_V4, num_leaves=255, learning_rate=0.05, min_data=150,
               rounds=1400, patience=120),
}

In [8]:
val_X, val_last, val_ys = build_split("val")
boosters = {}
for stride in (4, 2):
    Xtr, last_tr, ytr = build_split("train", stride)
    rng = np.random.default_rng(42)
    dead = Xtr[:, ALL_NAMES.index("allzero")] == 1
    keep = ~dead | (rng.random(len(dead)) < 0.15)
    Xtr = Xtr[keep]
    last_tr = last_tr[keep]
    ytr = {h: y[keep] for h, y in ytr.items()}
    print(f"stride {stride}: {Xtr.shape[0]:,} baris train")
    for name, cfg in VARIANTS.items():
        if cfg["stride"] != stride:
            continue
        cols = cfg["cols"]
        names = [ALL_NAMES[i] for i in cols]
        params = dict(BASE, num_leaves=cfg["num_leaves"], learning_rate=cfg["learning_rate"],
                      min_data_in_leaf=cfg["min_data"])
        Xc = Xtr if len(cols) == Xtr.shape[1] else Xtr[:, cols]
        Xvc = val_X if len(cols) == val_X.shape[1] else val_X[:, cols]
        for h in HORIZONS:
            t0 = time.time()
            dtr = lgb.Dataset(Xc, ytr[h] - last_tr, feature_name=names,
                              categorical_feature=[names.index("seg_id")], free_raw_data=True)
            dva = lgb.Dataset(Xvc, val_ys[h] - val_last, reference=dtr)
            bst = lgb.train(params, dtr, num_boost_round=cfg["rounds"], valid_sets=[dva],
                            callbacks=[lgb.early_stopping(cfg["patience"], verbose=False),
                                       lgb.log_evaluation(0)])
            boosters[(name, h)] = bst
            print(f"{name} h{h}: iter {bst.best_iteration}  "
                  f"val l2 {bst.best_score['valid_0']['l2']:.3f}  {time.time() - t0:.0f}s")
        del Xc, Xvc
        gc.collect()
    del Xtr
    gc.collect()

stride 4: 4,403,177 baris train


v1 h5: iter 195  val l2 24.729  47s


v1 h10: iter 207  val l2 28.200  50s


v1 h15: iter 82  val l2 31.437  30s


stride 2: 8,805,086 baris train


...

v3 h10: iter 199  val l2 27.893  162s


v3 h15: iter 122  val l2 31.039  123s


v4 h5: iter 252  val l2 24.512  154s


v4 h10: iter 191  val l2 27.885  135s


v4 h15: iter 109  val l2 31.064  109s


## Gabungkan dengan least squares

Bobot gabungan 4 model dihitung closed form di holdout untuk tiap horizon. Blend ini lebih baik dari semua model tunggal di ketiga horizon. Bobot negatif wajar muncul karena model saling berkorelasi.

In [9]:
def predict_variant(name, h, X):
    cols = VARIANTS[name]["cols"]
    Xc = X if len(cols) == X.shape[1] else X[:, cols]
    return boosters[(name, h)].predict(Xc)

weights = {}
for h in HORIZONS:
    y = val_ys[h].astype(np.float64)
    r_true = y - val_last
    R = np.stack([predict_variant(m, h, val_X) for m in VARIANTS], axis=1)
    for j, m in enumerate(VARIANTS):
        r = R[:, j]
        a = float(r_true @ r / (r @ r))
        print(f"h{h} {m}: mse {mse(np.clip(val_last + a * r, 0, 150), y):.3f}")
    w = np.linalg.solve(R.T @ R, R.T @ r_true)
    weights[h] = w
    blend = np.clip(val_last + R @ w, 0, 150)
    print(f"h{h} blend {np.round(w, 3)}: mse {mse(blend, y):.3f}")

h5 v1: mse 24.708
h5 v2: mse 24.573
h5 v3: mse 24.467
h5 v4: mse 24.492
h5 blend [0.091 0.076 0.463 0.339]: mse 24.427


h10 v1: mse 28.072
h10 v2: mse 28.121
h10 v3: mse 27.779
h10 v4: mse 27.790
h10 blend [ 0.135 -0.203  0.533  0.466]: mse 27.717


h15 v1: mse 31.325
h15 v2: mse 31.401
h15 v3: mse 30.909
h15 v4: mse 30.949
h15 blend [ 0.177 -0.388  0.672  0.469]: mse 30.813


## Prediksi test dan tulis submission

Prediksi akhir adalah nilai terakhir ditambah residual blend, dipotong ke rentang 0 sampai 150. Segmen yang seluruh window testnya nol dipaksa bernilai 0. Urutan id diverifikasi sama persis dengan sample submission.

In [10]:
txt_te = text_counts([texts["test"][f"test_{i:05d}"] for i in range(540)])
ev_te = {c: events["test"][c] for c in EV_CLASSES}
Xte = build_features(test_hist, txt_te, ev_te)
last_te = test_hist[:, -1, :].reshape(-1)
allzero = (test_hist == 0).all(axis=1).reshape(-1)

vals = np.empty((540, 3, N_SEG), dtype=np.float32)
for j, h in enumerate(HORIZONS):
    R = np.stack([predict_variant(m, h, Xte) for m in VARIANTS], axis=1)
    p = np.clip(last_te + R @ weights[h], 0, 150)
    p[allzero] = 0.0
    vals[:, j, :] = p.reshape(540, N_SEG)

ids = []
for i in range(540):
    for h in HORIZONS:
        base = f"test_{i:05d}_h{h}_r"
        ids.extend(base + str(r) for r in range(N_SEG))

sub = pd.DataFrame({"id": ids, "speed": vals.reshape(-1)})
sub.to_csv(f"{OUT}/submission_final.csv", index=False, float_format="%.3f")
sample = pd.read_csv(f"{DATA}/sample_submission.csv", usecols=["id"])
assert len(sample) == len(sub)
assert (sample["id"].values == sub["id"].values).all()
print(len(sub), "baris, urutan id cocok")

2041200 baris, urutan id cocok


## Hasil

MSE holdout blend final sekitar 24.4 untuk h5, 27.7 untuk h10, dan 30.8 untuk h15. Baseline persistence ada di kisaran 40 sampai 48 jadi perbaikannya sekitar 37%.